# 🛡️ EdgeShield: Small Language Model (SLM) Training Pipeline on Google Colab
**Doctoral Research Milestone | IEEE TEMSCON-ASPAC 2026 Accepted Paper**  
*Paper Title:* EdgeShield: A Small Language Model Framework for Real-Time Detection of Lateral Movement and Ransomware Attacks  
*Authors:* Gauravkumar Raval (Research Scholar) & Dr. Tejaskumar Bhatt (Supervisor)  
*Affiliation:* Faculty of Engineering & Technology, GLS University, Ahmedabad, India  

---

### 📌 What this Notebook Does:
1. **GPU Environment Verification**: Verifies NVIDIA CUDA GPU (T4 / A100 / V100).
2. **Repository & Dependencies**: Clones `slm-gpu-trainer` on `feat/edgeshield-slm-framework` and installs PyTorch/Transformers/PEFT stack.
3. **Automated Dataset Setup**: Generates curated ransomware traces, DARPA OpTC benchmark, and prepares LMD-2023 authentication telemetry.
4. **Stream A (LSA) Fine-Tuning**: Trains DeBERTa-v3-small / DistilBERT with Supervised Contrastive Loss & Cross-Entropy for Lateral Movement (TA0008).
5. **Stream B (BPD) Fine-Tuning**: Trains the SLM sequence classifier and Attention Threat Scoring Head for pre-encryption ransomware intercept (T1490, T1562.001, T1486).
6. **Sequence Classifier (v2 Sliding Window, K=3)**: Multi-class temporal sequence training with HuggingFace Trainer.
7. **Tier 2 Generative Reasoner (LoRA / QLoRA)**: Fine-tunes Qwen2.5-1.5B-Instruct to generate causal JSON incident reports.
8. **Dynamic INT8 Quantization**: Exports PyTorch weights to ONNX and dynamically quantizes to INT8 (11.71 ms latency, <50MB RAM).
9. **Empirical Evaluation**: Validates detection against XGBoost/LightGBM baselines and multi-stage attack scenarios.
10. **Google Drive Artifact Persistence**: Saves trained checkpoints and ONNX models directly to your Google Drive.


## 1. ⚙️ Check Hardware & GPU Acceleration
Ensure your Colab runtime is set to GPU. If needed, go to:
`Runtime` ➔ `Change runtime type` ➔ Select `T4 GPU` (or A100 if subscribed) ➔ `Save`.


In [ ]:
# Verify NVIDIA CUDA GPU and PyTorch environment
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    print(f"Device Count:     {torch.cuda.device_count()}")
    print(f"Allocated Memory: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")
    print(f"Reserved Memory:  {torch.cuda.memory_reserved(0)/(1024**2):.2f} MB")
    print("[+] Hardware acceleration is ACTIVE and ready for training!")
else:
    print("[!] WARNING: No CUDA GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.")
print("=" * 60)


## 2. 📂 Clone Repository & Set Working Directory
Clones the active `feat/edgeshield-slm-framework` branch and enters the `slm-tr` working directory.


In [ ]:
import os

REPO_URL = "https://github.com/graval/slm-gpu-trainer.git"
BRANCH = "feat/edgeshield-slm-framework"
TARGET_DIR = "/content/slm-gpu-trainer"

if not os.path.exists(TARGET_DIR):
    print(f"[*] Cloning repository from {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL} {TARGET_DIR}
else:
    print(f"[*] Repository already present at {TARGET_DIR}. Pulling latest changes...")
    !cd {TARGET_DIR} && git pull origin {BRANCH}

# Switch working directory to slm-tr
os.chdir(os.path.join(TARGET_DIR, "slm-tr"))
print(f"[+] Active working directory: {os.getcwd()}")
!pwd && ls -F


## 3. 📦 Install Dependencies
Installs required packages for Hugging Face Transformers, Datasets, PEFT, TRL, ONNX Runtime, and BitsAndBytes.


In [ ]:
# Install required libraries silently
!pip install -q --upgrade pip
!pip install -q transformers>=4.38.0 datasets>=2.14.0 accelerate>=0.28.0 peft>=0.9.0 trl>=0.8.0
!pip install -q scikit-learn pandas numpy onnx onnxruntime onnxscript bitsandbytes optimum-quanto
!pip install -q sentencepiece tiktoken safetensors altair colorama

print("[+] All dependencies successfully installed!")


## 4. 📊 Dataset Preparation
Prepares the datasets for both Stream A and Stream B:
* **Curated Ransomware Traces (`ransomware_behavioral_traces.csv`)**: High-fidelity traces across T1490, T1562.001, and T1486.
* **DARPA OpTC Out-of-Distribution Benchmark (`optc_test_benchmark.csv`)**: eCAR host telemetry test set.
* **Authentication Telemetry (`lmd_2023_dataset.csv`)**:
  - Automatically builds authentic security telemetry from OTRF Mordor if the 1GB file is not provided.
  - Or load your full 1.75M records LMD-2023 CSV from Google Drive.


In [ ]:
import os
import sys

# Ensure slm-tr is in python path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

os.makedirs("data", exist_ok=True)

# 1. Generate Curated Ransomware Behavioral Traces for BPD
print("[*] Generating Curated Ransomware Behavioral Traces for BPD...")
from edgeshield.data.synthetic_ransomware import generate_curated_ransomware_dataset
bpd_df = generate_curated_ransomware_dataset("data/ransomware_behavioral_traces.csv", num_samples=12000)
print(f"[+] BPD dataset ready: {len(bpd_df):,} samples generated.")

# 2. Generate DARPA OpTC Benchmark
print("[*] Generating DARPA OpTC Benchmark dataset...")
from scripts.prepare_optc_benchmark import generate_optc_test_benchmark
generate_optc_test_benchmark("data/optc_test_benchmark.csv", total_samples=1500)

# 3. Check for LMD-2023 dataset or compile authentic Mordor telemetry fallback
lmd_path = "data/lmd_2023_dataset.csv"
if not os.path.exists(lmd_path):
    print("[*] data/lmd_2023_dataset.csv not detected.")
    print("[*] Compiling authentic security telemetry from OTRF Mordor dataset as baseline...")
    from scripts.setup_dataset import generate_fallback_dataset
    generate_fallback_dataset()
else:
    print(f"[+] Active LMD-2023 dataset detected ({os.path.getsize(lmd_path):,} bytes).")

print("\n[+] Current datasets in data/ directory:")
!ls -lh data/


### Step 4B: Acquire the Full 1.07 GB LMD-2023 Dataset (2 Options)

The peer-reviewed LMD-2023 dataset contains 1,752,836 Sysmon events (~1.07 GB). You can provide it using either:

* **Option 1 (Fastest): Mount your personal Google Drive**
  If you uploaded `lmd_2023_dataset.csv` from your local machine to your Google Drive (`MyDrive/lmd_2023_dataset.csv`), mount Drive and copy it in 5 seconds.
* **Option 2: Direct download from author's public Google Drive folder using `gdown`**
  Downloads directly from the published author folder (`1PkJiGpD0Kn1rV8GC9m2b_eWvqQTSa1eH`).


In [ ]:
# OPTION 1: Mount Google Drive and copy from your personal drive:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/lmd_2023_dataset.csv" data/lmd_2023_dataset.csv
# print(f"[+] LMD-2023 copied from Drive: {os.path.getsize('data/lmd_2023_dataset.csv'):,} bytes")

# OPTION 2: Download directly from author's public Google Drive link using gdown:
# !pip install -q gdown
# !gdown --folder "https://drive.google.com/drive/folders/1PkJiGpD0Kn1rV8GC9m2b_eWvqQTSa1eH?usp=sharing" -O data/
# if os.path.exists("data/LMD-2023-labeled.csv") and not os.path.exists("data/lmd_2023_dataset.csv"):
#     os.rename("data/LMD-2023-labeled.csv", "data/lmd_2023_dataset.csv")
# !ls -lh data/


## 5A. 🎯 Train Stream A: Log Semantic Analyzer (LSA)
Fine-tunes the SLM backbone (`microsoft/deberta-v3-small` or `distilbert-base-uncased`) with **Joint Supervised Contrastive Loss & Cross-Entropy** ($L_{\text{total}} = L_{\text{CE}} + \lambda L_{\text{SCL}}$) for Windows authentication & Lateral Movement detection (MITRE ATT&CK TA0008).


In [ ]:
# Execute Stream A Fine-Tuning on CUDA GPU
!python scripts/train_edgeshield_lsa.py \
    --model_name "microsoft/deberta-v3-small" \
    --dataset_path "data/lmd_2023_dataset.csv" \
    --device "cuda" \
    --epochs 3 \
    --batch_size 16 \
    --lr 5e-5 \
    --use_contrastive \
    --num_samples 12000 \
    --test_samples 2000 \
    --output_dir "models/edgeshield_lsa_deberta"


## 5B. ⚡ Train Stream B: Behavioral Pattern Detector (BPD)
Fine-tunes the SLM sequence classifier and **Attention Threat Scoring Head** on ransomware behavioral telemetry (T1490, T1562.001, T1486) for pre-encryption intercept.


In [ ]:
# Execute Stream B Fine-Tuning on CUDA GPU
!python scripts/train_edgeshield_bpd.py \
    --model_name "distilbert-base-uncased" \
    --dataset_path "data/ransomware_behavioral_traces.csv" \
    --device "cuda" \
    --epochs 3 \
    --batch_size 16 \
    --threat_weight 0.5 \
    --num_samples 8000 \
    --test_samples 1500 \
    --output_dir "models/edgeshield_bpd"


## 5C. 🔄 (Optional) Train Sequence Classifier (v2 Sliding Window, K=3)
Trains the multi-class lateral movement classifier across sliding temporal windows ($K=3$) using Hugging Face Trainer on CUDA GPU.


In [ ]:
# Train v2 Sliding Window Classifier with HuggingFace Trainer
!python train_classifier.py \
    --variant v2 \
    --window_size 3 \
    --device cuda \
    --epochs 3 \
    --batch_size 16 \
    --model_name "microsoft/deberta-v3-small" \
    --csv_path "data/lmd_2023_dataset.csv" \
    --output_dir "models/deberta-lateral-movement-v2_sliding_window"


## 5D. 🧠 (Optional) Train Tier 2 Generative Reasoner (LoRA / QLoRA)
Fine-tunes `Qwen/Qwen2.5-1.5B-Instruct` with Parameter-Efficient Fine-Tuning (PEFT LoRA) and 4-bit quantization (QLoRA) to generate causal JSON incident reports and automated remediation steps.


In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"

# Train Generative Reasoner with 4-bit QLoRA on CUDA GPU
!python train_generator.py \
    --model_name "Qwen/Qwen2.5-1.5B-Instruct" \
    --variant v2 \
    --window_size 3 \
    --csv_path "data/lmd_2023_dataset.csv" \
    --epochs 1 \
    --batch_size 2 \
    --gradient_accumulation_steps 4 \
    --learning_rate 2e-4 \
    --qlora \
    --output_dir "models/qwen-lateral-movement-v2_sliding_window"


## 6. 🚀 ONNX Export & Dynamic INT8 Quantization
Compresses the trained PyTorch weights from 176MB to **44MB** (4x reduction) and accelerates inference to **11.71 ± 0.62 ms** on edge hardware.


In [ ]:
# Export PyTorch model to ONNX and dynamically quantize to INT8
!python scripts/export_onnx.py \
    --checkpoint "models/edgeshield_lsa_deberta" \
    --out-dir "models/onnx_edgeshield" \
    --name "edgeshield_lsa" \
    --quantize

# Benchmark ONNX INT8 latency across 500 runs
!python scripts/benchmark_onnx.py \
    --model "models/onnx_edgeshield/edgeshield_lsa_int8.onnx" \
    --runs 500 \
    --export_summary


## 7. 📈 Benchmark Empirical Results & Baselines
Evaluates multi-stage attack scenarios (ALPHV/BlackCat, LockBit 3.0 Pass-the-Hash) and validates performance against baseline classifiers (XGBoost, Random Forest, LightGBM).


In [ ]:
# Run full empirical benchmark across 500 samples
!python scripts/benchmark_edgeshield.py --samples 500 --export_summary

# View generated benchmark summary
import json
if os.path.exists("external/edgeshield_benchmark_summary.json"):
    with open("external/edgeshield_benchmark_summary.json") as f:
        print(json.dumps(json.load(f), indent=2))


## 8. 💾 Persist Models to Google Drive
Saves all trained checkpoints, LoRA adapters, ONNX INT8 models, and JSON summaries into your Google Drive so they are safely preserved after the Colab session terminates.


In [ ]:
from google.colab import drive
import os

print("[*] Mounting Google Drive...")
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/EdgeShield_Colab_Artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"[*] Copying trained models to Google Drive: {SAVE_DIR} ...")
!cp -r models/* "{SAVE_DIR}/"
if os.path.exists("external"):
    !cp -r external/*.json "{SAVE_DIR}/" 2>/dev/null || true

print(f"[+] All artifacts successfully saved to: {SAVE_DIR}")
!ls -lh "{SAVE_DIR}"
